# Solar Rooftop Potential — Raster Approach

Compute solar thermal and PV potential for a region using a **built-up area raster** (GeoTIFF) to identify building footprints. This approach works anywhere a land-cover raster is available.

---

## 1. Imports

In [ ]:
import atlite
import hvplot.pandas
import matplotlib.pyplot as plt
import numpy as np
import rasterio as rio
import xarray as xr

from src.config import load_config
from src.data_loading import load_cop, load_cutout, load_demand, load_nuts, load_shape
from src.solar_thermal import compute_solar_thermal

hvplot.extension("bokeh")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})


## 2. File Paths

All input data lives in `data/`. Adjust paths below if you are using your own data.

In [ ]:
# Try loading from config; fall back to defaults if config.yaml is absent
try:
    cfg = load_config("config.yaml")
    ks = str(cfg.resolve_path(cfg.region.nuts))
    shape_path = str(cfg.resolve_path(cfg.region.shapefile))
    built_area_path = str(cfg.resolve_path(cfg.data.builtup_raster))
    demand_path = str(cfg.resolve_path(cfg.data.demand))
    cop_path = str(cfg.resolve_path(cfg.data.cop))
    year = cfg.parameters.year
    cap_per_sqkm = cfg.parameters.capacity_density
    alpha_values = cfg.parameters.alpha_values
    buffer = cfg.parameters.buffer
    cutout_path = cfg.resolve_path(
        cfg.data.cutout.format(year=year, region=cfg.region.name)
    )
except FileNotFoundError:
    cfg = None
    ks = "data/NUTS_RG_10M_2021_4326.geojson"
    shape_path = "data/leeste3035.gpkg"
    built_area_path = "data/IBU_2018_010m_eu_03035_V1_0.tif"
    demand_path = "data/Last_240222.csv"
    cop_path = "data/COP_WP.xlsx"
    year = 2014
    cap_per_sqkm = 8  # MW/km² (raster approach)
    alpha_values = [0, 0.2, 0.4, 0.6, 0.8, 1]
    buffer = 0.005
    cutout_path = "data/era5-2014-leeste.nc"
    from src.config import ParametersConfig
    st_params = ParametersConfig(year=year, capacity_density=cap_per_sqkm)

nuts = load_nuts(ks)


## 3. Region Shape & Exclusion Container

Load the region boundary, set up an atlite `ExclusionContainer` using the built-up area raster, and define the ERA5 cutout extent. Using `nuts.total_bounds` gives a wide buffer so the weather data covers the full region.

In [ ]:
shape = load_shape(shape_path)
shape = shape.set_index("Name")

buildings_excluder = atlite.ExclusionContainer(crs=3035, res=10)
buildings_excluder.add_raster(built_area_path, crs=3035, codes=1, invert=True)

minx, miny, maxx, maxy = nuts.total_bounds
x_slice = slice(minx - buffer, maxx + buffer)
y_slice = slice(miny - buffer, maxy + buffer)

SHAPE_INDEX_NAME = shape.index[0]


## 4. Load Demand & COP Data

`load_demand()` reads the CSV, converts kW → MW, and returns separate Series for domestic hot water (DHW) and space heating (SH). `load_cop()` reads COP values for air-source heat pumps.

In [ ]:
load_dhw, load_sph = load_demand(demand_path, year=year)
cop = load_cop(cop_path, year=year)


## 5. ERA5 Weather Cutout

Create/reuse an atlite `Cutout`. If `era5-2014-leeste.nc` does not exist, uncomment `cutout.prepare()` to download from the CDS API (requires API key — see `docs/setup_cds_api.md`).

In [ ]:
cutout = load_cutout(cutout_path, x_slice, y_slice, year=year)

# cutout.prepare()   # uncomment to download ERA5 data


## 6. Availability Matrix

`cutout.availabilitymatrix()` returns the fraction of each grid cell that is available for solar installation. A capacity density (`cap_per_sqkm`) converts this to MW of installable capacity. The map below shows the available area in green.

In [ ]:
A = cutout.availabilitymatrix(shape, buildings_excluder)

area = cutout.grid.set_index(["y", "x"]).to_crs(3035).area / 1e6
area = xr.DataArray(area, dims=("spatial",))
capacity_matrix = A.stack(spatial=["y", "x"]) * area * cap_per_sqkm

band, transform = atlite.gis.shape_availability(shape.geometry, buildings_excluder)

fig, ax = plt.subplots(figsize=(8, 8))
shape.plot(ax=ax, color="none")
ax.set_xlabel("Eastings (m)")
ax.set_ylabel("Northings (m)")
rio.plot.show(band, transform=transform, cmap="Greens", ax=ax)
plt.title("Available Area for Solar Installation")
plt.show()


## 7. Solar Thermal Potential

Two flat-plate collector models are available, selectable via `st_model` in the config:
- **`atlite`** (default) — Henning & Palzer (2014) linear efficiency model, uses `cutout.solar_thermal()`
- **`oemof`** — EN 12975 quadratic efficiency model (same as oemof.thermal)

Both use direct + diffuse irradiance and temperature-dependent efficiency, a significant improvement over the earlier CSP-based approximation.

In [ ]:
st_potential = compute_solar_thermal(
    cutout, shape, capacity_matrix,
    params=cfg.parameters if cfg is not None else st_params,
    orientation="latitude_optimal",
)

coverage_st = st_potential.sum() / load_dhw.sum() * 100
print(f"Solar thermal covers {coverage_st:.1f}% of DHW demand")


## 8. PV Potential

Atlite simulates **CdTe thin-film PV panels** at latitude-optimal tilt. The result is the hourly PV electricity generation from all available roof area in the region.

In [ ]:
pv = cutout.pv(
    panel=atlite.solarpanels.CdTe,
    matrix=capacity_matrix,
    orientation="latitude_optimal",
    index=shape.index,
)
pv = pv.to_pandas()
pv = pv[SHAPE_INDEX_NAME]

coverage_pv = pv.sum() / load_sph.sum() * 100
print(f"PV covers {coverage_pv:.1f}% of space heating demand (via HP)")


## 9. PV-Powered Heat Pump

Multiplying hourly PV generation by the hourly COP gives the **thermal output of air-source heat pumps** powered by the PV panels. This is the heat delivered for space heating.

In [ ]:
hp = pv * cop


## 10. Time-Series Plots

### 10.1 Solar Thermal Potential

In [ ]:
st_potential.plot.line(
    linewidth=0.5, figsize=(14, 4),
    xlabel="Time", ylabel="Solar Thermal Potential (MW)",
    title="Solar Thermal Potential"
)
plt.show()


### 10.2 Domestic Hot Water Demand

In [ ]:
load_dhw.plot.line(
    linewidth=0.5, figsize=(14, 4),
    xlabel="Time", ylabel="DHW Demand (MW)",
    title="Domestic Hot Water Load"
)
plt.show()


### 10.3 Solar Thermal — DHW Mismatch

Positive values = overproduction (excess heat that cannot be stored or must be discarded).

In [ ]:
diff1 = st_potential - load_dhw
diff1.plot.area(
    figsize=(14, 4), xlabel="Time",
    ylabel="Mismatch (MW)",
    title="Solar Thermal to DHW: Production minus Consumption"
)
plt.show()

diff1.resample("D").mean().plot.area(
    figsize=(14, 4), xlabel="Time",
    ylabel="Mismatch (MW)",
    title="Daily Average Mismatch"
)
plt.show()


### 10.4 PV Potential

In [ ]:
pv.plot.line(
    linewidth=0.5, figsize=(14, 4),
    xlabel="Time", ylabel="PV Potential (MW)",
    title="PV Potential"
)
plt.show()


### 10.5 Heat Pump Thermal Output

In [ ]:
hp.plot.line(
    linewidth=0.5, figsize=(14, 4),
    xlabel="Time", ylabel="Thermal Power (MW)",
    title="Air-Source Heat Pump Thermal Output (PV-powered)"
)
plt.show()


### 10.6 Space Heating Demand

In [ ]:
load_sph.plot.line(
    linewidth=0.5, figsize=(14, 4),
    xlabel="Time", ylabel="Space Heating Load (MW)",
    title="Space Heating Load"
)
plt.show()


### 10.7 Heat Pump — Space Heating Mismatch

In [ ]:
diff2 = hp - load_sph
diff2.plot.area(
    figsize=(14, 4), xlabel="Time",
    ylabel="Mismatch (MW)",
    title="HP Production minus Space Heating Demand"
)
plt.show()

diff2.resample("D").mean().plot.area(
    figsize=(14, 4), xlabel="Time",
    ylabel="Mismatch (MW)",
    title="Daily Average Mismatch"
)
plt.show()


## 11. Sensitivity Analysis ($\alpha$ Variation)

Vary the allocation ratio $\alpha$ between PV and solar thermal:
- $\alpha = 0$ → all roof area to solar thermal
- $\alpha = 1$ → all roof area to PV

For each $\alpha$, we compute load coverage and overproduction.

In [ ]:
st_a, hp_a = {}, {}
demand_cover_dw_a, demand_cover_sh_a = {}, {}

for a in alpha_values:
    pv_temp = pv * a
    st_temp = st_potential * (1 - a)
    hp_temp = pv_temp * cop

    diff1_t = st_temp - load_dhw
    diff2_t = hp_temp - load_sph

    st_a[a] = diff1_t[diff1_t > 0].sum()
    hp_a[a] = diff2_t[diff2_t > 0].sum()

    demand_cover_dw_a[a] = st_temp.sum() / load_dhw.sum() * 100
    demand_cover_sh_a[a] = hp_temp.sum() / load_sph.sum() * 100

print("Overproduction DHW (MWh):", st_a)
print("Overproduction SH  (MWh):", hp_a)
print("DHW coverage       (%):", demand_cover_dw_a)
print("SH  coverage       (%):", demand_cover_sh_a)


### 11.1 Load Coverage Plot

In [ ]:
keys = alpha_values
v1 = list(demand_cover_dw_a.values())
v2 = list(demand_cover_sh_a.values())

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(keys, v1, alpha=0.5, label="Domestic Hot Water")
ax.fill_between(keys, v2, alpha=0.5, label="Space Heating")
ax.set_xlabel("$\\alpha$")
ax.set_ylabel("Load Coverage [%]")
ax.set_title("Sensitivity Analysis — Load Coverage")
ax.legend(loc="upper left")
ax.grid(True)
plt.show()
# fig.savefig("sensitivity_coverage.pdf", dpi=300)


### 11.2 Overproduction Plot

In [ ]:
v3 = np.array(list(st_a.values())) / load_dhw.sum() * 100
v4 = np.array(list(hp_a.values())) / load_sph.sum() * 100 / cop.mean()

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(keys, v3, alpha=0.5, label="Domestic Hot Water")
ax.fill_between(keys, v4, alpha=0.5, label="PV")
ax.set_xlabel("$\\alpha$")
ax.set_ylabel("Overproduction / Yearly Demand [%]")
ax.set_title("Sensitivity Analysis — Overproduction")
ax.legend(loc="upper left")
ax.grid(True)
plt.show()
# fig.savefig("sensitivity_overproduction.png", dpi=300)
